# Hello Spark Basics

Welcome to your first Spark notebook! This notebook introduces the fundamentals of Apache Spark and PySpark.

## Learning Objectives

By the end of this notebook, you will:
- Understand what Spark is and why it's useful
- Create your first SparkSession
- Create simple DataFrames
- Perform basic operations
- Understand the Spark UI

## What is Apache Spark?

Apache Spark is a unified analytics engine for large-scale data processing. It provides:

- **Speed**: In-memory processing, up to 100x faster than Hadoop MapReduce
- **Ease of Use**: APIs in Python, Scala, Java, R, and SQL
- **Generality**: SQL, streaming, machine learning, and graph processing
- **Runs Everywhere**: YARN, Mesos, Kubernetes, standalone, or in the cloud

### Key Concepts

- **RDD (Resilient Distributed Dataset)**: The original Spark abstraction (low-level)
- **DataFrame**: A distributed collection of data organized into named columns (high-level, recommended)
- **Dataset**: Type-safe version of DataFrame (Scala/Java only)
- **SparkSession**: The entry point for Spark functionality

## 1. Creating a SparkSession

The SparkSession is the entry point to all Spark functionality. It replaces the older SQLContext and HiveContext.

In [1]:
import sys
sys.path.insert(0, "/opt/spark")

from utils.connect_session import get_connect_url
from pyspark.sql import SparkSession

# Spark Connect: this notebook is a thin client.
# All computation runs on the Spark cluster (see Spark UI).
spark = (
    SparkSession.builder
    .appName("Hello-Spark-Basics")
    .remote(get_connect_url())   # e.g. sc://spark-master:15002
    .getOrCreate()
)


Spark Version: 3.5.5
Connected to: sc://spark-master:15002


### Understanding the SparkSession Builder

- `.appName()`: Sets a name for your application (visible in Spark UI)
- `.getOrCreate()`: Gets existing session or creates a new one

You can also configure:
- `.master()`: Set the cluster URL (e.g., `local[*]` for local mode)
- `.config()`: Set Spark configuration properties

In [2]:
# This notebook is a Spark Connect CLIENT — the master URL lives on the server side.
# See where your work actually runs in the Spark UI:
print("Spark UI:      http://localhost:4040 (single-node mode)")
print("Master UI:     http://localhost:8180 (cluster mode)")
print("History Server: http://localhost:18080")
print("All jobs from this notebook appear there, exactly like spark-submit jobs.")


Master: local[*]


## 2. Creating Your First DataFrame

DataFrames are the primary abstraction in Spark. Think of them as distributed tables with rows and columns.

In [2]:
# Create a DataFrame from a list of tuples
data = [
    ("Alice", 25, "Engineering"),
    ("Bob", 30, "Marketing"),
    ("Charlie", 35, "Engineering"),
    ("Diana", 28, "Sales"),
    ("Eve", 32, "Engineering")
]

columns = ["name", "age", "department"]

df = spark.createDataFrame(data, columns)

# Show the DataFrame
df.show()

+-------+---+-----------+
|   name|age| department|
+-------+---+-----------+
|  Alice| 25|Engineering|
|    Bob| 30|  Marketing|
|Charlie| 35|Engineering|
|  Diana| 28|      Sales|
|    Eve| 32|Engineering|
+-------+---+-----------+



In [4]:
# Examine the schema
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- department: string (nullable = true)



### Understanding the Schema

The schema defines the structure of your data:
- `name`: StringType (text data)
- `age`: LongType (integer, inferred as Long)
- `department`: StringType

Spark infers the schema automatically, but you can also define it explicitly for more control.

In [5]:
# Define schema explicitly
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), False),  # False = not nullable
    StructField("age", IntegerType(), True),   # True = nullable
    StructField("department", StringType(), True)
])

df_with_schema = spark.createDataFrame(data, schema)
df_with_schema.printSchema()

root
 |-- name: string (nullable = false)
 |-- age: integer (nullable = true)
 |-- department: string (nullable = true)



## 3. Basic DataFrame Operations

Let's explore some fundamental DataFrame operations.

In [3]:
# Select specific columns
df.select("name", "age").show()

+-------+---+
|   name|age|
+-------+---+
|  Alice| 25|
|    Bob| 30|
|Charlie| 35|
|  Diana| 28|
|    Eve| 32|
+-------+---+



In [4]:
# Filter rows
df.filter(df.age > 30).show()

+-------+---+-----------+
|   name|age| department|
+-------+---+-----------+
|Charlie| 35|Engineering|
|    Eve| 32|Engineering|
+-------+---+-----------+



In [6]:
# Group by and count
df.groupBy("department").count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|Engineering|    3|
|  Marketing|    1|
|      Sales|    1|
+-----------+-----+



In [7]:
# Sort by age (descending)
df.orderBy(df.age.desc()).show()

+-------+---+-----------+
|   name|age| department|
+-------+---+-----------+
|Charlie| 35|Engineering|
|    Eve| 32|Engineering|
|    Bob| 30|  Marketing|
|  Diana| 28|      Sales|
|  Alice| 25|Engineering|
+-------+---+-----------+



In [10]:
# Add a new column
from pyspark.sql.functions import col, lit

df_with_bonus = df.withColumn(
    "bonus",
    col("age") * 100  # Bonus is age * 100
)
df_with_bonus.show()

+-------+---+-----------+-----+
|   name|age| department|bonus|
+-------+---+-----------+-----+
|  Alice| 25|Engineering| 2500|
|    Bob| 30|  Marketing| 3000|
|Charlie| 35|Engineering| 3500|
|  Diana| 28|      Sales| 2800|
|    Eve| 32|Engineering| 3200|
+-------+---+-----------+-----+



## 4. Lazy Evaluation

One of Spark's key concepts is **lazy evaluation**. Transformations are not executed until an action is called.

- **Transformations**: `select()`, `filter()`, `groupBy()`, `groupBy().count()` - return a new DataFrame
- **Actions**: `show()`, `df.count()`, `collect()` - trigger computation

> ⚠️ **Important**: `count()` behaves differently depending on context:
> - `df.count()` → **Action** (returns an integer, triggers computation)
> - `df.groupBy().count()` → **Transformation** (returns a DataFrame, no computation)

In [13]:
# count() behaves differently depending on context!

# count() as an ACTION - returns integer, triggers computation
total_rows = df.count()  # Computation happens here!
print(f"Total rows (df.count() is an action): {total_rows}")
print(f"Return type: {type(total_rows)}\n")

# count() as a TRANSFORMATION - returns DataFrame, no computation yet
grouped_df = df.groupBy("department").count()  # Just creates a plan
print(f"grouped_df type: {type(grouped_df)}")
print("No computation happened yet - this is a transformation!")

Total rows (df.count() is an action): 5
Return type: <class 'int'>

grouped_df type: <class 'pyspark.sql.dataframe.DataFrame'>
No computation happened yet - this is a transformation!


In [14]:
# groupBy().count() is a TRANSFORMATION - creates a plan, doesn't execute
transformed_df = df \
    .filter(col("age") > 25) \
    .select("name", "department") \
    .groupBy("department") \
    .count()

print("Transformation defined, but not yet executed!")
print("Spark has created a logical plan.")

Transformation defined, but not yet executed!
Spark has created a logical plan.


In [15]:
# Now the action triggers execution
transformed_df.show()
print("Now the computation happened!")

+-----------+-----+
| department|count|
+-----------+-----+
|  Marketing|    1|
|Engineering|    2|
|      Sales|    1|
+-----------+-----+

Now the computation happened!


## 5. Reading and Writing Data

Spark can read from and write to many data sources.

In [16]:
# Write to Parquet (recommended format)
# NOTE: write to the shared mount /opt/spark/data — in cluster mode,
# a bare /tmp path is container-local per node, so the driver could
# not read files the executors wrote.
df.write.mode("overwrite").parquet("/opt/spark/data/tmp/spark_basics/people.parquet")

# Read it back
df_read = spark.read.parquet("/opt/spark/data/tmp/spark_basics/people.parquet")
df_read.show()

+-------+---+-----------+
|   name|age| department|
+-------+---+-----------+
|Charlie| 35|Engineering|
|  Alice| 25|Engineering|
|    Eve| 32|Engineering|
|    Bob| 30|  Marketing|
|  Diana| 28|      Sales|
+-------+---+-----------+



In [ ]:
# Also supports CSV, JSON, and more
df.write.mode("overwrite").csv("/opt/spark/data/tmp/spark_basics/people.csv", header=True)
df.write.mode("overwrite").json("/opt/spark/data/tmp/spark_basics/people.json")

print("Data written in multiple formats!")

## 6. The Spark UI

When you run Spark, a web UI is available to monitor your jobs.

- **Single-node mode**: http://localhost:4040
- **Cluster mode**: http://localhost:8080 (Master UI)

The UI shows:
- Jobs and their status
- Stages and tasks
- Executor memory usage
- SQL query plans

**Exercise**: Open the Spark UI in another tab and observe what happens when you run the next cell.

In [17]:
# This will show up in the Spark UI
result = df \
    .filter(col("age") > 25) \
    .groupBy("department") \
    .agg({"age": "avg"}) \
    .collect()  # This is an action

print(f"Result: {result}")
print("\nCheck the Spark UI to see the job that just ran!")

Result: [Row(department='Marketing', avg(age)=30.0), Row(department='Engineering', avg(age)=33.5), Row(department='Sales', avg(age)=28.0)]

Check the Spark UI to see the job that just ran!


## 7. Exercises

Try these exercises to reinforce your learning:

In [ ]:
# Exercise 1: Create a DataFrame with 10 rows of product data
# Columns: product_id, product_name, price, category
# Your code here:


In [ ]:
# Exercise 2: Filter products with price > 50 and sort by price descending
# Your code here:


In [ ]:
# Exercise 3: Calculate the average price per category
# Your code here:


## Summary

In this notebook, you learned:

1. How to create a SparkSession
2. How to create DataFrames from Python data
3. Basic DataFrame operations (select, filter, groupBy, orderBy)
4. The concept of lazy evaluation
5. How to read and write data
6. How to monitor jobs in the Spark UI

### Next Steps

- **Notebook 2**: DataFrames and SQL - deeper dive into DataFrame API
- **Notebook 3**: Transformations - map, filter, joins
- **Examples**: Check `examples/01-fundamentals/` for more scripts

In [18]:
# Clean up
spark.stop()
print("SparkSession stopped. Goodbye!")

SparkSession stopped. Goodbye!
